# report03 — 표적 검증 — 우리 드론 모형을 실물·커뮤니티 3D 모델과 대조한다

**핵심.** 검출 실험에 넣을 **자작 파라메트릭 드론 메쉬**를, 시뮬레이터 밖의 실물 CAD·커뮤니티 3D 모델에 겹쳐 스스로 검증한다 — 평균 밝기(σ) 눈금은 세 기체 모두 ±1 dB 로 맞고, 방향별 뾰족값(6~12 dB 차이)은 인용하지 않는다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | Sionna RT 는 표적 메쉬를 받아 전파를 계산하지만 **그 메쉬가 실물을 닮았는지는 검증하지 않는다.** 파라메트릭 메쉬의 형상 충실도를 보장할 표준 절차가 스톡 파이프라인엔 없다 — 공식 치수(L×W×H) 준수는 필요조건일 뿐(치수표 숫자 몇 개를 지켰다는 것) 표적 충실도의 증명이 아니다. |
| **② 선행 연구의 방식** | 선행은 표적 충실도를 **파이프라인 밖의 독립 기준**에 앵커한다 — 외부 EM/CAD 로 계산한 UAV RCS 를 가져오거나(LAMBDA=Sionna+CADFEKO, arXiv:2607.03826; Temporal-GNN=점산란체, arXiv:2604.08306), 무향실 실측 드론 RCS(NCSU Ezuma·Güvenç · BUPT 3GPP unified RCS)로 눈금을 맞춘다. '표적은 바깥 기준으로 검증한다'가 표준(§4). |
| **③ 쓴 라이브러리·결합** | **trimesh** 로 외부 실물 CAD·커뮤니티 메쉬를 로드하고 우리 표적 메쉬와 **같은 좌표계로 정렬·오버레이**해 투영면적·형상을 겹쳐 본다(널리 쓰는 메쉬 라이브러리를 재사용, 중복 구현 없음). 밝기 σ 는 선행과 같은 SBR+PO 로 계산해 비교한다(§5). |
| **④ 검증** | 제조사 실물 CAD(Yuneec Typhoon H480)·커뮤니티 메쉬(M100·M600) 오버레이 + 주요 치수(대각·프롭경) 대조. 평균 σ 차이 +0.92·+0.51·+0.92 dB(±1 dB), 방향별 RMS 6~12 dB. 절대 눈금은 실측 문헌 RCS 로 report08. |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| Yuneec Typhoon H480 **실물 CAD** (동체) | ethz-asl/rotors_simulator (Apache-2.0). `benchmark/compare_real_cad.py` 가 우리 파라메트릭과 대조 | 제조사 실물 형상 (오픈 라이선스). **우리가 만들지 않은 외부 모형** |
| Holybro 1345 **프로펠러 실물 CAD** | PX4/PX4-gazebo-models (BSD-3). 우리 NACA-4 익형 프롭과 대조 | 실물 부품 CAD. 부품 단위 보조 점검 |
| DJI Matrice 100·600 Pro **커뮤니티 메쉬** | github: TareqAlqutami/dji_ros_simulator (DJI Onboard-SDK 시뮬용). `benchmark/compare_community.py` 가 대조 | 커뮤니티 3D 모델 (구형 기체). **방법 점검용** |
| Gazebo/PX4 모델 조사 | `docs/GAZEBO_PX4_MODELS.md` — PX4-gazebo-models · PX4-SITL_gazebo-classic · 커뮤니티 저장소 목록 | 공개 저장소 조사 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `trimesh-cad` | CAD 모델링 (`src/cadkit.py` + `src/drone_cad.py`) — 로프트·스윕·회전체·**불리언(CSG)** | 🔴 **별도** (trimesh + manifold3d + shapely + scipy, CPU) |
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 대조는 CPU 로 방위 각도마다 그림자 투영·표면 적분을 도는 수 초~수 분 규모. 종합 그림은 CPU 초 단위. (표적 밝기 σ 의 본 계산 물리는 → report06.)

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 실물 CAD 대조 (Typhoon H480 + Holybro 1345 프롭) → outputs/real_cad_compare.json
~/.venvs/py312/bin/python benchmark/compare_real_cad.py

# 커뮤니티 메쉬 대조 (M100 · M600) → outputs/community_compare.json
~/.venvs/py312/bin/python benchmark/compare_community.py

# 신뢰 경계 종합 그림 + 인터넷 mesh vs 자작 mesh 시각 비교
~/.venvs/py312/bin/python src/viz_report3_cad.py
~/.venvs/py312/bin/python src/viz_report3_overlay.py

# Gazebo/PX4 SDF 내보내기
~/.venvs/py312/bin/python src/gazebo_export.py

# 노트북 재생성
~/.venvs/py312/bin/python src/make_notebook03.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/real_cad_compare.json` | Typhoon H480·Holybro 프롭 대조의 모든 숫자 (방위별 σ·면적·평균차) |
| `outputs/community_compare.json` | Matrice 100·600 대조의 모든 숫자 |
| `outputs/figures/report03_confidence.png` | 신뢰 경계 종합 그림 (어디까지 믿나) |
| `outputs/figures/report03_mesh_overlay.png` | 인터넷 mesh vs 자작 mesh 시각 비교(형상+σ) |
| `outputs/gazebo/<key>/model.sdf` | 드론 5종의 Gazebo/PX4 비행 시뮬 모델 (SDF) |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **세 잣대는 우리와 다른 기체다.** Typhoon H480 은 Yuneec(DJI 아님) 헥사, M100·M600 은 DJI 이지만 2015~2016 **구형**이다. 우리 표적(Mavic 4 Pro·Matrice 4E 등)은 2025 신형이라 공개된 실물 3D 모델이 아직 없다. → 이 대조가 검증하는 것은 **'우리 방법이 대략 맞나'이지 '이 특정 신형 기체가 정확하다'가 아니다.**
- **방향별(각도별) σ 뾰족값은 인용 금지.** 평균은 ±1 dB 로 맞지만 각도별 RMS 는 6~12 dB 어긋난다. 널·글린트의 **위치**는 잔가지 형상에 민감해 재현되지 않는다. 우리는 **평균 밝기와 그 분포**만 인용한다.
- **투영면적은 체계적으로 작다**(커뮤니티 메쉬 대비 -3.7~-3.0 dB). 착륙다리·GPS 마스트 같은 가는 돌출부를 덜 그린 결과다. 알려진 편향이며, 실측이 아니라 모형 간 상대비교다.
- **Gazebo/PX4 SDF 는 형상·질량까지다.** 실제로 '날리려면' 관성텐서·추력계수·모터 시상수 등 동역학 파라미터가 더 필요하고, 그 상당수는 어느 제조사도 공개하지 않아 추정이다. 이 리포트의 태스크(디텍션)에는 비행 시뮬이 필수가 아니다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **report02** (드론 제작) | 앞 리포트 — 여기서 대조할 그 모형을 스펙시트에서 깎았다. 그 리포트의 '외형 0.00 % 일치'가 왜 증명이 아닌지를 이 리포트가 이어받는다 |
| **report04** (조명 파형) | 다음 리포트 — 믿을 만한 표적을 세웠으니, 이제 그 표적을 **무엇으로 비출지**(상시 통신 신호)로 넘어간다 |
| **report06** (RCS·SBR 물리) | 여기서 '평균 밝기 눈금이 맞다'고 확인한 그 σ 를 어떻게 계산하는지의 물리를 다룬다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **RCS (σ)** | 레이더 되비침 밝기 [m²]. '이 표적이 얼마나 밝게 되쏘는가'. dBsm = 10·log₁₀(σ/1 m²) |
| **투영면적** | 표적을 한 방향에서 봤을 때의 그림자 넓이. 클수록 대체로 더 밝게 되쏜다 |
| **CAD** | 컴퓨터로 그린 정밀 3D 도면. 여기선 제조사·커뮤니티가 공개한 진짜 3D 모델을 뜻한다 |
| **메쉬(mesh)** | 3D 형상을 삼각형 그물로 표현한 것. 정점 + 면 |
| **파라메트릭 모형** | 치수·곡률 같은 파라미터로 규칙에 따라 깎은 우리 모형(앞 리포트에서 제작) |
| **널(null)** | 특정 각도에서 되쏨이 서로 상쇄돼 밝기가 뚝 떨어지는 지점 |
| **글린트(glint)** | 특정 각도에서 되쏨이 겹쳐 밝기가 확 튀는 지점(거울에 햇빛 반짝하듯) |
| **RMS 차이** | 각도별 오차를 제곱평균한 값. 방향별로 얼마나 들쭉날쭉 어긋나는지의 척도 |
| **PO** | 물리광학 — 빛 닿은 표면 조각의 되쏨을 위상 맞춰 적분해 밝기를 낸다 |
| **SBR** | 광선 쏘고 튕기기 — 표적을 광선으로 조준해 보이는 면을 찾고 그 위에서 PO 적분(가림 포함) |
| **SDF** | Simulation Description Format — Gazebo/PX4 가 읽는 로봇·기체 모델 파일 |
| **dBsm** | dB relative to 1 m². 밝기 σ 를 로그 눈금으로 적은 것 |

</details>

---


---

**앞 리포트**에서 드론 5종의 3D 모형을 세웠고, 그 외형은 DJI 공식 L×W×H 에 맞춰져 있다. 이 모형이 그대로 Sionna RT 의 표적 메쉬로 들어간다. 다만 **치수표 준수는 필요조건일 뿐, 표적 충실도의 증명은 아니다** — 시뮬레이터는 넣은 메쉬가 실물을 닮았는지 검증하지 않기 때문이다. 그래서 이 리포트는 표적을 **시뮬레이터 밖의 독립 기준(외부 실물 CAD·커뮤니티 3D 모델·측정 RCS 문헌)** 과 대봄으로써 검증한다 — 선행 연구가 표적을 검증하는 바로 그 방식이다.

## §1. Sionna 의 공백 — 넣은 메쉬가 실물을 닮았는지 검증되지 않는다

전파 시뮬레이터에 넣는 표적 메쉬는 시뮬레이터가 **검증해 주지 않는다.** 외형을 공식 치수(전체 길이·폭·높이)에 맞추는 것은 필요조건이지만, 그것이 보장하는 것은 딱 하나 — **'치수표의 숫자 몇 개를 지켰다'** 뿐이다.

치수표에 없는 것 — 동체의 곡률, 짐벌의 모양, 팔의 굵기, 착륙다리의 유무 — 은 치수표만으로 정해지지 않는다. 그런데 레이더 밝기(RCS)는 바로 그 세부에 반응한다. 따라서 **공식 치수를 지킨 파라메트릭 메쉬라도 실물 형상을 닮았다는 보장은 없고**, 그 형상 충실도를 담보할 표준 절차가 스톡 파이프라인엔 없다.

그래서 표적 충실도는 **파이프라인 밖의 독립 기준**으로 세운다. 이 리포트는 우리와 아무 관계 없는 **외부 3D 모델(실물 CAD·커뮤니티 메쉬)**과 겹쳐 봄으로써 그 빠진 절차를 채운다(선행이 이 기준을 어디서 찾는지는 §4).

## §2. 무엇을·어떻게 대보나 — 세 외부 기준과 두 잣대

우리와 무관하게 **공개된** 3D 모델 세 개를 모았다. 셋 다 우리 파이프라인 밖에서, 다른 사람이 다른 목적으로 만든 것이다. 형상 라이브러리 **trimesh** 로 이들을 읽어 우리 표적 메쉬와 같은 좌표계에 정렬해 겹쳐 본다.

| 잣대 | 기체 | 성격 | 규모/복잡도 |
|---|---|---|---|
| **A. 실물 CAD** | Typhoon H480 (real CAD, Apache-2.0) | 제조사(Yuneec) 헥사콥터의 **실물 3D 도면**, 오픈 라이선스 | 면 178,840 vs 18,832 |
| **B. 커뮤니티** | DJI Matrice 100 (2015 quad) (community mesh) | 커뮤니티가 만든 구형 DJI 쿼드 메쉬 | 대각 650 mm · 프롭 342 mm |
| **C. 커뮤니티** | DJI Matrice 600 Pro (2016 hexa) (community mesh) | 커뮤니티가 만든 구형 DJI 헥사 메쉬 | 대각 1133 mm · 프롭 533 mm |

**왜 구형·타사인가.** 우리의 실제 표적은 Mavic 4 Pro·Matrice 4E 같은 **2025 신형**이다. 신형은 너무 새로워서 **공개된 실물 3D 모델이 세상에 아직 없다** — 선행의 측정 드론 RCS 문헌도 대부분 구형 기체다. 그래서 구할 수 있는 가장 가까운 외부 기준 — 타사 실물 CAD(Typhoon)와 구형 DJI 커뮤니티 메쉬(2015·2016) — 로 대신한다. 참고로 Matrice 100(구형, 대각 650 mm)과 우리 표적 Matrice 4E(신형, 대각 439 mm)는 **완전히 다른 기체**다. 이 대조가 검증하는 것은 **'스펙시트 → 형상 → 밝기' 파이프라인의 눈금이 맞나**이지, '이 특정 신형 기체가 정확하다'가 아니다.

*(부품 단위 보조 점검으로 Holybro 1345 prop (real CAD, BSD-3) 도 NACA-4 익형 프롭과 대봤다 — 아래 종합 그림 맨 오른쪽 막대.)*

![.](outputs/renders/anim/spin_matrice4e.gif)

<sub>Matrice 4E 3D 모델 회전 — 실물·커뮤니티 CAD 로 파이프라인을 교차검증한 실측 실험용 드론.</sub>

**두 잣대로 잰다.** 같은 방향(방위각 0°→360°, 낮은 앙각 el=15°, 3.5 GHz)에서 두 모형을 각각 조명해 두 값을 잰다:

1. **겉모양 = 투영면적.** 그 방향에서 본 표적의 **그림자 넓이**. 크면 대체로 더 밝게 되쏜다. 순수 기하 — 형상만 다르면 값이 갈린다.
2. **밝기 = RCS(σ).** 표면 조각들이 위상을 맞춰 되쏘는 양을 다 더한 값. 선행(예: BVH SBR+PO, arXiv:2604.09243)이 쓰는 것과 같은 **SBR+PO** 방식 — 광선으로 보이는 면을 찾고(SBR) 그 위에서 물리광학 적분(PO) — 으로 계산한다. 형상뿐 아니라 표면의 곡률·정렬까지 반영한다. (밝기 계산의 물리 상세는 report06·07, 절대값 앵커는 report08.)

그리고 두 모형의 차이를 **두 가지 방식으로** 요약한다 — 이 구분이 이 리포트의 핵심이다:

- **평균 차이 (Δ평균 σ)**: 360° 전체를 평균한 밝기가 얼마나 다른가. → **눈금이 맞는가**를 본다.
- **방향별 RMS 차이**: 각도마다의 밝기가 얼마나 들쭉날쭉 어긋나는가. → **세부 패턴이 맞는가**를 본다.

다음 §3 에서 보겠지만 **이 둘의 성적표가 완전히 다르다** — 그리고 그 차이가 '무엇을 인용하고 무엇을 인용하지 않을지'를 정한다.

## §3. 증거 — 세 외부 기준에 겹쳐 본 결과

먼저 **눈으로** 겹쳐 본다. 왼쪽 = 인터넷에서 가져온 실물/커뮤니티 3D 모델, 오른쪽 = 우리가 그 기체의 **스펙시트로 만든 것**(같은 축척·같은 시점).

![internet mesh vs our spec-built mesh](outputs/figures/report03_mesh_overlay.png)

> ⚠ **이 그림의 핵심은 '형상이 똑같다'가 아니다.** 이들은 우리 신형 타깃이 아니라 **공개 모델이 존재하는 다른 기체**(Yuneec Typhoon·구형 DJI M100/M600)이고, 커뮤니티 모델은 프로펠러를 **스윕 디스크**(회전면)로 그리는 등 제작 방식도 제각각이다. 그래서 세부 형상은 당연히 다르게 보인다. 우리가 확인하는 것은 하나 — **같은 로터 수·같은 크기의 기체를 우리 파이프라인으로 지었을 때 되쏘는 평균 밝기(RCS)가 외부 모델과 ±1 dB 로 맞느냐**(각 기체 제목의 Δσ)이다.

<sub>양쪽 모두 PEC(완전도체)로 통일해 형상만 본다(재질은 report08 에서 실측으로 앵커). 형상 세부가 다르므로 **각도별** RCS 는 어긋나고(6~12 dB, 아래 결과②), 그래서 우리는 **평균만** 인용한다. 이 대조의 값어치는 '우리 방법을 우리와 무관한 외부 실기체 3개로 독립 검증했다'는 것 자체다.</sub>

### 결과 ① 평균 밝기는 세 기체 모두 ~1 dB 이내로 맞는다

| 기체 | 진짜 모델 평균 σ | 우리 모형 평균 σ | 차이 (우리 − 진짜) |
|---|---|---|---|
| Typhoon H480 | -16.96 dBsm | -16.04 dBsm | **+0.92 dB** |
| Matrice 100 (2015 쿼드) | -14.29 dBsm | -13.78 dBsm | **+0.51 dB** |
| Matrice 600 Pro (2016 헥사) | -9.16 dBsm | -8.24 dBsm | **+0.92 dB** |

세 기체의 평균 밝기 차이가 모두 **±1 dB 근방**(+0.92 · +0.51 · +0.92 dB)이다. 제조사 실물 CAD(A)와 커뮤니티 메쉬(B·C), 쿼드와 헥사, 다른 크기·다른 회사 — 그런데도 평균은 일관되게 붙는다.

세 잣대는 서로 무관하게, 서로 다른 사람이, 서로 다른 기체를 모델링한 것이다. 만약 SBR+PO 방식에 체계적 눈금 오류가 있었다면 세 대조에서 **같은 방향으로 크게** 어긋났을 것이다. 그런데 셋 다 1 dB 안쪽이라는 것은, **'스펙시트 → 형상 → SBR+PO 밝기' 파이프라인이 평균적으로 올바른 양의 에너지를 되쏜다**는 뜻이다.

이 대조는 **모형끼리의 상대비교**로 방식의 눈금을 본다. 이 평균 σ 의 **절대값**이 무향실에서 실제로 잰 동종 멀티로터 드론 RCS 문헌(NCSU Ezuma·Güvenç, BUPT 3GPP unified RCS) 범위 안에 드는지 — 즉 절대 눈금까지 맞는지 — 는 **report08 에서 실측 RCS 앵커**로 확인한다. → 그래서 우리는 표적의 **평균 밝기 σ 를 인용한다.**

### 결과 ② 방향별 뾰족값은 6~12 dB 어긋난다 (그래서 인용하지 않는다)

평균은 붙었지만, **특정 각도에서의** 밝기는 얘기가 다르다:

| 기체 | 방향별 RMS 차이 |
|---|---|
| Typhoon H480 | **8.9 dB** |
| Matrice 100 | **8.2 dB** |
| Matrice 600 Pro | **11.6 dB** |
| (참고) Holybro 1345 프롭 | 6.4 dB |

각도별로 보면 **6~12 dB** 나 들쭉날쭉 어긋난다(종합 그림 아래 **B**). 반면 σ 값 자체의 **분포**는 서로 가깝다 — 종합 그림 맨 윗줄의 **누적분포(CDF)** 에서 실선(진짜 모델)과 점선(우리 모형)이 대체로 겹치고 평균선(세로 점선)이 거의 만난다. **평균 레벨과 분포는 맞지만, 뾰족한 급락(널)·급등(글린트)이 어느 각도에 오는지는 서로 안 맞는다.**

이 뾰족한 봉우리와 골짜기는 **간섭** 때문이다. 표면의 여러 조각이 되쏜 파동이 어떤 각도에선 마루끼리 겹쳐 확 밝아지고(글린트), 어떤 각도에선 마루와 골이 만나 서로 지워진다(널). 이건 **잔잔한 호수에 돌 여러 개를 던졌을 때** 생기는 물결 무늬와 같아서 — 돌의 위치를 몇 cm만 옮겨도 무늬가 만나는 지점(밝고 어두운 자리)이 확 바뀐다.

두 모형은 **잔가지 형상이 다르다**(곡률·이음매·돌기). 그 미세한 차이가 '돌의 위치'를 조금 옮기고, 그 결과 널·글린트가 **어느 각도에 오는지**가 통째로 달라진다. **전체 물결 에너지(=평균 밝기)는 보존되지만, 무늬의 위치는 재현되지 않는다.**

그래서 우리는 '이 각도에서 σ 는 몇 dBsm' 같은 방향별 뾰족값을 **인용하지 않는다.** 인용하는 것은 **평균 밝기와 그 분포(널이 얼마나 깊고 얼마나 자주 오는지의 통계)** 까지다. 이건 특정 모형만의 약점이 아니라 **어떤 근사 형상이든 갖는 성질**이다 — 실물과 나사 하나까지 같지 않으면 뾰족값의 위치는 옮겨간다. 선행의 측정 드론 RCS 연구가 σ 를 각도별 값이 아니라 **통계 분포(예: GEV·Swerling 요동 모델)** 로 다루는 것도 같은 이유다.

### 결과 ③ 모형은 가는 돌출부를 덜 그려 살짝 홀쭉하다

| 기체 | 투영면적 차이 (우리 − 진짜) | 방향 |
|---|---|---|
| Matrice 100 (커뮤니티) | **-3.71 dB** | 우리가 작다 |
| Matrice 600 Pro (커뮤니티) | **-3.03 dB** | 우리가 작다 |
| Typhoon H480 (실물 CAD) | +1.83 dB | 우리가 살짝 크다 |
| (참고) Holybro 1345 프롭 | -0.15 dB | 우리가 작다 |

커뮤니티 실측 메쉬 두 개에서 우리 모형의 그림자 넓이가 **3.0~3.7 dB 작다.** 원인은 분명하다 — **착륙다리·GPS 안테나 마스트·튀어나온 센서** 같은 **가는 돌출부가 덜 그려져 있다.** Matrice 100 커뮤니티 메쉬는 위로 솟은 GPS 마스트와 아래로 뻗은 착륙다리를 세밀히 담고 있지만, 파라메트릭 실루엣은 그런 잔가지를 단순화한다.

(Typhoon 실물 CAD 에서는 반대로 우리가 +1.83 dB **크게** 나왔다 — 실물 CAD 는 접이식 팔·틈이 있어 그림자에 빈 곳이 많은데, 매끈한 동체가 그 실루엣을 더 꽉 채우기 때문이다. 방향에 따라 ±2 dB 안팎으로 갈린다.)

이건 '어느 쪽이 더 정직한 밝기냐'의 판정이 아니라, 모형의 **알려진 편향**으로 기록해 두는 것이다 — 가는 돌출부를 덜 그린다 → 투영면적이 조금 작다 → 그 방향의 밝기도 그만큼 보수적(작게)일 수 있다. 실측 안테나 측정이 아니라 **모형끼리의 상대비교**라는 점도 함께 기억해 둔다.

### 한 장으로 보는 신뢰 경계 — 어디까지 믿고, 어디부터 못 믿나

![신뢰 경계](outputs/figures/report03_confidence.png)

**읽는 법:**
- **맨 윗줄 (누적분포 CDF 3개)**: 방위별 σ 값의 **분포**. **파란 실선 = 진짜 모델, 빨간 점선 = 우리 모형.** 두 분포가 서로 가깝고 평균선(세로 점선)이 거의 겹친다(결과 ①) → **레벨은 인용 가능.** 각도별 뾰족값의 위치 불일치(결과 ②)는 아래 B 가 정량화한다.
- **아래 A**: **평균 밝기 차이.** 초록 띠가 ±1 dB 안전 구간 — 세 기체 막대가 모두 그 안.
- **아래 B**: **방향별 RMS 차이.** 6~12 dB — 방향별 뾰족값은 인용 불가.
- **아래 C**: **투영면적 차이.** 커뮤니티 메쉬에서 우리가 3.0~3.7 dB 작음 — 가는 돌출부 미모델링.

| 무엇을 | 성적 | 인용해도 되나 |
|---|---|---|
| 평균 밝기 σ (전방위 평균) | ±1 dB 일치 | ✅ **인용** |
| 밝기의 분포·통계 (널의 깊이·빈도) | 대체로 일치 | ✅ 통계로 인용 |
| **방향별 σ 뾰족값** (이 각도 = 몇 dBsm) | 6~12 dB 차이 | ❌ **인용 금지** |
| 투영면적 (겉모양 넓이) | 우리가 3.0~3.7 dB 작음 | ⚠️ 알려진 편향으로만 |

## §4. 선행 연구는 표적 충실도를 어떻게 세우나

이 한계(시뮬레이터가 표적 메쉬를 검증하지 않는다)를 다루는 방식은 우리만의 것이 아니다. Sionna·RF 디지털트윈을 센싱에 쓰는 선행은 표적 충실도를 **파이프라인 밖의 독립 기준**에 앵커한다:

- **외부 EM/CAD 로 계산한 UAV RCS 를 가져와** 채널에 주입한다 — LAMBDA(Sionna+CADFEKO UAV RCS, arXiv:2607.03826), Temporal-GNN(점산란체 RCS 주입, arXiv:2604.08306). 표적 산란만 외부 물리로 구해 두 전파 구간 사이에 끼워 넣는 구조($h=h_{bg}+h_{target}$)다.
- **무향실에서 실제로 잰 드론 RCS 문헌으로 눈금을 맞춘다** — NCSU Ezuma·Güvenç, BUPT 3GPP unified RCS 등. 이들은 σ 를 각도별 값이 아니라 **통계 분포**로 제공한다(결과 ②에서 본 이유와 같다).

요컨대 **'표적은 모델 파이프라인과 무관한 바깥 기준으로 검증한다'가 이 분야의 표준 관행**이다. 이 리포트는 바로 그 표준을 따라, 우리와 아무 관계 없는 외부 실물 CAD·커뮤니티 메쉬에 겹쳐 본다.

## §5. 우리가 쓴 방식 — trimesh 오버레이 + SBR+PO, 그리고 검증

**형상 대조는 표준 메쉬 라이브러리 `trimesh` 로 한다** — 외부 실물 CAD·커뮤니티 메쉬를 로드하고, 우리 표적 메쉬와 같은 좌표계로 정렬해 겹친 뒤, 방위각을 돌리며 투영면적(그림자 넓이)을 잰다. 새 기하 커널을 짜지 않고 널리 쓰이는 라이브러리를 그대로 재사용하므로 중복 구현이 없다.

**밝기(σ)는 선행이 쓰는 것과 같은 SBR+PO 로 계산해 비교한다** — 광선으로 보이는 면을 찾고(SBR) 그 위에서 물리광학 적분(PO). 선행의 RCS 처리 세 갈래(① 상용 full-wave CADFEKO=LAMBDA · ② 자작 SBR+PO=BVH SBR+PO arXiv:2604.09243 · ③ 점산란체/UTD 근사) 중 **②**를 따른 것으로, 상용툴 없이 재현 가능하다.

**검증 요약 — 무엇을 어디까지 확인했나:**

| 확인 | 방법 | 결과 |
|---|---|---|
| 형상 충실도 | 실물 CAD(Typhoon H480)·커뮤니티 메쉬(M100·M600) 오버레이 | 겹침 확인 |
| 평균 밝기 눈금 | 세 기체 평균 σ 차이 | **+0.92·+0.51·+0.92 dB** (±1 dB) |
| 방향별 패턴 | 방향별 RMS 차이 | **6~12 dB** → 평균만 인용 |
| 투영면적 편향 | 커뮤니티 메쉬 대비 | **-3.7~-3.0 dB** (우리가 작다) |

이 리포트의 검증은 **모형끼리의 상대비교**로 방법의 눈금을 세운다. σ 의 **절대 눈금**(무향실 실측 문헌 RCS 앵커)은 **report08**, 밝기 계산의 물리 상세는 **report06·07** 소관이다.

## §6. 보너스 — 드론을 비행 시뮬(Gazebo/PX4)로 내보내기

표적 모형은 앞 리포트에서 **몸통과 프로펠러를 따로 움직이는 관절 구조**로 세워졌다. 그 구조는 비행 시뮬레이터 Gazebo 가 원하는 '동체 링크 하나 + 로터마다 회전 관절 하나'와 **같은 형태**라, 드론 5종을 그대로 표준 모델 파일 **SDF**(Gazebo/PX4 가 읽는 형식)로 내보낼 수 있다 (→ `outputs/gazebo/<key>/model.sdf`).

조사 결과(`docs/GAZEBO_PX4_MODELS.md`):
- 공식 PX4/Gazebo 에 **완제품 DJI 드론은 없다.** DJI 는 PX4 가 아니라 자체 비행스택을 쓰기 때문이다.
- 커뮤니티 DJI 모델은 있지만 전부 **구형**(Matrice 100·600·Phantom 4·Tello)이고, PX4 가 아니라 DJI 자체 SDK 시뮬용이다.
- **Mavic 4 Pro·Matrice 4E(2025 신형)는 어디에도 없다** — 너무 새로워서.

→ 즉 이 내보내기는 **세상에 준비된 적 없는 신형 기체의 비행 모델** 빈틈을 채운다. 각 부위 밀도로 관성텐서를, 호버 조건에서 추력계수를, 볼록껍질로 충돌메쉬를 붙여 SDF 를 채운다.

다만 디텍션에는 이게 필수가 아니다. 레이더가 보는 것(표적의 밝기·마이크로도플러)은 이미 Sionna RT + SBR 이 담당한다. Gazebo/PX4 가 더해 줄 수 있는 건 **현실적인 비행 궤적**뿐인데, 디텍션 실험에는 직선/저속 궤적이면 충분하다. 실제로 '날리려면' SDF 형상 위에 관성·추력계수·모터 시상수 같은 **동역학 파라미터**가 더 필요하고, 그 상당수는 어느 제조사도 공개하지 않아 추정에 기댄다. 그래서 비행 시뮬은 **열어 둔 선택지**로 남겨 둔다.

## 정리 & 다음

### 답한 질문: *검출 실험에 넣을 이 드론 모형을 얼마나 믿어도 되나?*

표적 충실도를 시뮬레이터 밖의 독립 기준으로 세웠다 — 우리와 무관하게 공개된 3D 모델 **세 개**(실물 CAD Typhoon H480 + 커뮤니티 M100·M600)를 trimesh 로 겹쳐 보고, 밝기는 선행이 쓰는 SBR+PO 로 비교했다. 프로펠러는 Holybro 1345 실물 CAD 로 보조 점검했다.

| 주장 | 근거 | 값 |
|---|---|---|
| 평균 밝기 눈금이 맞다 | 세 기체 평균 σ 차이 | **+0.92 · +0.51 · +0.92 dB** (모두 ±1 dB) |
| 방향별 뾰족값은 못 믿는다 | 방향별 RMS 차이 | **6~12 dB** → 평균만 인용 |
| 우리 모형이 살짝 홀쭉하다 | 투영면적 차이 | 커뮤니티 대비 **-3.7~-3.0 dB** |

**믿으면 안 되는 것** (앞머리 8️⃣ 참조): 방향별 σ 뾰족값(널·글린트 위치) · 세 잣대가 신형과 같은 기체라는 착각(구형/타사) · 투영면적을 실측 밝기로 오해하기.

**핵심 한 줄**: 우리 모형은 **평균 밝기의 눈금은 믿어도 되고, 방향별 뾰족값과 가는 돌출부는 믿으면 안 된다.** 그리고 이 대조는 신형 한 대의 완벽함이 아니라 **방법의 타당성**을 재는 것이다.

| 다음 리포트 | 무엇을 이어받나 |
|---|---|
| **report04** — 조명 파형 | 믿을 만한 표적을 세웠으니, 이제 그 표적을 **무엇으로 비출지**(상시 통신 신호)로 넘어간다. |
| **report06** — RCS·SBR 물리 | 여기서 '평균 눈금이 맞다'고 확인한 그 σ 를 **어떻게 계산하는지**의 물리를 다룬다. |